<style>
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2 { color: #003399; }
.jp-RenderedHTMLCommon h3 { color: #1f4e79; }
.note { border-left: 4px solid #003399; padding: 0.4em 0.9em; background: #f4f7fb; }
.warning { border-left: 4px solid #d97706; padding: 0.4em 0.9em; background: #fff8eb; }
details { margin: 0.7em 0; padding: 0.3em 0.7em; background: #f7f7f7; }
</style>

# Практика: зарплаты в вакансиях

У нас есть вакансии специалистов по данным. Для каждой будущей вакансии нужно предсказать среднюю зарплату в тысячах долларов в год.

Главная сложность здесь не в том, чтобы один раз получить маленькую ошибку. Нужно построить проверку, которой можно верить, понять, каким признакам можно доверять, заметить изменение данных со временем и извлечь полезную информацию из текста.

В начале доступны только `train.csv` с ответами и `test.csv` без ответов. Размеченный test откроем один раз в конце, когда решение уже зафиксировано.


## Что понадобится

Понадобится знакомство с `pandas`, линейной регрессией, train/validation/test и RMSE.

Работаем как на олимпиаде: каждое решение принимаем по validation, а не по будущему test.


## Олимпиадный чеклист

В этой практике мы последовательно применим идеи из лекции к одной задаче. Этот список можно потом использовать и в других турах:

- понять объект, target и момент прогноза;
- проверить происхождение признаков;
- выбрать validation, похожую на неизвестный test;
- получить простой ориентир и сравнивать train с validation;
- проверить представление даты и придумать новые признаки;
- следить, какие преобразования обучаются только на train;
- проверить, нужна ли сложной модели регуляризация;
- зафиксировать решение до открытия test.

Перед каждым экспериментом сформулируйте три вещи: какую гипотезу вы проверяете, какой результат будет аргументом в ее пользу и что может сделать такой результат обманчивым. Длинные ответы записывать не нужно, достаточно обсудить идею вслух.

Открывайте подсказку только после собственной попытки. Подсказки помогают с инструментами, но не выбирают решение за вас.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings

from scipy.sparse import hstack
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 30)

warnings.filterwarnings(
    "ignore",
    message=r"(divide by zero|overflow|invalid value) encountered in matmul",
    category=RuntimeWarning,
    module=r"sklearn\.linear_model\._base",
)

TARGET = "salary_k_usd"


## 0. Загружаем данные

Загрузим два файла: размеченный train и будущий test без ответов. Ответы test откроем только в конце.


In [ ]:
TRAIN_URL = "https://drive.google.com/uc?export=download&id=1OLBotfoA5oJpS3KyKVpqEWue-lbcqMJ7"
TEST_URL = "https://drive.google.com/uc?export=download&id=17IAX47AY3A67BWzu25D192TyR7PqEieO"

train_full = pd.read_csv(TRAIN_URL, parse_dates=["posting_date"])
test = pd.read_csv(TEST_URL, parse_dates=["posting_date"])

print("train:", train_full.shape)
print("test:", test.shape)


In [ ]:
display(train_full.head(3).drop(columns="job_description"))
print(train_full.dtypes)


Одна строка соответствует одной вакансии. Прогноз строится в день публикации, поэтому в модель можно передавать только информацию, известную в этот момент.

Target `salary_k_usd` есть только в train. Это средняя годовая зарплата в тысячах долларов.


## Проверка 1. Одинаково ли устроены train и test

В олимпиадной задаче сначала сравнивают сами таблицы. Target должен быть только в train. Найдите остальные столбцы, которые есть лишь в одной из выборок.

<details>
<summary>Подсказка</summary>

Столбцы можно превратить в `set` и вычесть одно множество из другого.

</details>


In [ ]:
# Выведите столбцы, которые есть только в train и только в test.
# TARGET рассматривайте отдельно.


Можно ли построить финальный прогноз по признаку, которого нет в test? Посмотрите на смысл найденных названий.


### Сравнение числовых распределений

Одинаковые названия столбцов еще не означают одинаковые данные. Для каждого общего числового признака сравните среднее и стандартное отклонение. Разность средних разделите на типичный масштаб признака.

<details>
<summary>Подсказка</summary>

Пригодятся [`select_dtypes`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html), [`mean`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mean.html), [`std`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.std.html) и `pd.DataFrame`.

В знаменателе можно использовать среднее двух стандартных отклонений. Добавьте небольшое число, чтобы не делить на ноль.

</details>


In [ ]:
# Соберите numeric_shift со столбцами:
# train_mean, test_mean, train_std, test_std, standardized_difference.
# Отсортируйте признаки по standardized_difference.


### Сдвиг категориальных распределений

Для категорий сравним все доли сразу. Половина суммы абсолютных различий долей называется total variation distance:

$$
\mathrm{TVD}(p,q)=\frac{1}{2}\sum_k |p_k-q_k|.
$$

TVD равна `0` для одинаковых распределений и `1`, если множества значений не пересекаются. Посчитайте ее для общих категориальных признаков, где не больше ста разных значений.

<details>
<summary>Подсказка</summary>

Для каждого столбца соедините два результата `value_counts(normalize=True)` через `pd.concat(..., axis=1)`, заполните отсутствующие доли нулями и просуммируйте модуль разности.

</details>


In [ ]:
# Соберите categorical_shift со столбцами feature и tvd.
# Отсортируйте признаки по убыванию tvd.


### Распределения отдельных признаков

Две таблицы показывают кандидатов, но форму распределения лучше увидеть. Постройте гистограммы `rating` и сравните доли значений `company_size`: это лидеры среди простых числовых и категориальных проверок.

<details>
<summary>Подсказка</summary>

Для числового признака подойдет [`hist`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html). Для категориального используйте [`value_counts(normalize=True)`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html), соедините две серии через `pd.concat` и вызовите `.plot.bar()`.

</details>


In [ ]:
# Постройте на одном рисунке распределения rating в train и test.
# Затем сравните доли значений company_size.


Какие различия видны на графиках? Сформулируйте гипотезу: сможет ли модель отличить train от test без даты?


### Модель «train или test»

Ручные графики показывают только выбранные признаки. Более общий способ поиска сдвига: объединить таблицы и обучить классификатор предсказывать источник строки.

Прямой признак даты не используем: test целиком находится в будущем, и по дате задача решалась бы без анализа остальных различий. Зато добавим длину описания, чтобы модель могла заметить простой сдвиг в текстах.

ROC AUC около `0.5` означает, что выборки трудно отличить. Чем ближе AUC к `1`, тем заметнее сдвиг.

Масштабирование и one-hot должны обучаться заново внутри каждого фолда. Для этого соберем преобразования и модель в `Pipeline`.

<details>
<summary>Подсказка</summary>

Документация: [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) и [`StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html).

Добавьте метку `is_test`: `0` для train и `1` для test. В `ColumnTransformer` числа передайте в `StandardScaler`, а категории в `OneHotEncoder`.

</details>

<div class="note">В соревновании такая проверка возможна, потому что признаки test уже даны. В реальном сервисе будущие объекты часто еще недоступны.</div>


In [ ]:
# Объедините train и test в source_data и добавьте is_test.
# Не используйте posting_date и train-only столбцы.
# Добавьте description_length и сохраните списки SOURCE_NUMERIC
# и SOURCE_CATEGORICAL.


In [ ]:
# Соберите source_preprocessor через ColumnTransformer.
# Затем соберите source_model: снача преобразования,
# затем LogisticRegression.


In [ ]:
# Посчитайте ROC AUC по пяти фолдам.
# После проверки обучите source_model на всех строках.


In [ ]:
# Достаньте имена признаков и коэффициенты из Pipeline.
# Выведите самые большие положительные и отрицательные коэффициенты.


Что говорит AUC о переносе модели? Какие различия подтверждаются и графиками, и коэффициентами классификатора?


## Проверка 2. Как признаки связаны с ответом

Начнем с числовых корреляций. Будущая validation начинается 1 февраля 2025 года, поэтому для первичного анализа используйте только более ранние строки. Найдите корреляцию каждого числового столбца с `salary_k_usd` и отсортируйте значения по убыванию модуля.

<details>
<summary>Подсказка</summary>

Сначала выберите числовые столбцы методом [`select_dtypes`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html). Затем пригодятся [`corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html), `abs` и [`sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_values.html).

</details>


In [ ]:
# Создайте correlation_data из строк до 1 февраля 2025 года.
# Выведите числовые признаки в порядке убывания
# абсолютной корреляции с TARGET.


Корреляция дала кандидатов для проверки, но не ответ. Сначала разберем признаки, которых нет в test. Проверьте, насколько середина двух числовых границ близка к target.

<details>
<summary>Подсказка</summary>

Создайте `midpoint_k` как полусумму `salary_floor_k` и `salary_ceiling_k`. Для проверки пригодятся корреляция, максимальная абсолютная ошибка и диаграмма рассеяния.

</details>


In [ ]:
# Создайте midpoint_k.
# Сравните его с TARGET численно и на графике.


Как объяснить почти идеальную диагональ? Как `salary_estimate` связан с двумя числовыми границами? Что произойдет с validation, если оставить хотя бы одно из этих представлений ответа?

Теперь проверьте `industry_salary_mean_all`. Восстановите среднюю зарплату каждой отрасли по всему train и сравните ее с готовым столбцом.

<details>
<summary>Подсказка</summary>

Пригодится `train_full.groupby("industry")[TARGET].transform("mean")`. Метод `transform` вернет значение для каждой исходной строки.

</details>


In [ ]:
# Создайте reconstructed_industry_mean по всему train.
# Найдите максимальное отличие от industry_salary_mean_all.
# Постройте диаграмму рассеяния двух значений.


Вопросы:

1. Что означает почти нулевое отличие восстановленного среднего от готового столбца?
2. Почему использование такого столбца в validation дает модели информацию об ответах validation?
3. Можно ли исправить среднее по отрасли, если считать его только по прошлым вакансиям?
4. Почему высокая корреляция сама по себе не является причиной удалить признак?

Посмотрите также на текстовые столбцы: числовая таблица корреляций их не показывает.


In [ ]:
# По результатам своего разбора составьте DROP_COLUMNS.
# Затем создайте train_clean и test_clean без выбранных столбцов.

DROP_COLUMNS = [
    # ...
]


In [ ]:
assert TARGET in train_clean.columns
assert TARGET not in test_clean.columns
assert not set(DROP_COLUMNS) & set(train_clean.columns)
assert list(train_clean.drop(columns=TARGET).columns) == list(test_clean.columns)

print("После удаления:", train_clean.shape, test_clean.shape)


Не продолжайте, пока не можете коротко объяснить, почему каждый столбец оказался в `DROP_COLUMNS`. Высокая корреляция сама по себе не является достаточной причиной.


## Проверка 3. Как устроен неизвестный test

Посмотрите на периоды train и test. До кода решите, чем validation должна отличаться от train, чтобы ее результат что-то говорил о будущем test.


In [ ]:
print("train:", train_clean["posting_date"].min(), "-", train_clean["posting_date"].max())
print("test:", test_clean["posting_date"].min(), "-", test_clean["posting_date"].max())


Test целиком находится в будущем относительно train. Поэтому validation тоже должна быть последним непрерывным отрезком времени: вакансии с 1 февраля по 30 июня 2025 года. Более ранние строки останутся в train.

Отсортируйте таблицу по дате и выполните такое разбиение.

<details>
<summary>Подсказка</summary>

Пригодятся [`sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html), `pd.Timestamp` и фильтрация вида `table[table["date"] < border]`.

</details>


In [ ]:
# Создайте train_part и validation_part.
# Не используйте test.


In [ ]:
assert len(train_part) + len(validation_part) == len(train_clean)
assert train_part["posting_date"].max() < validation_part["posting_date"].min()
assert train_part[TARGET].notna().all()
assert validation_part[TARGET].notna().all()

print("train:", train_part.shape, train_part["posting_date"].min(), train_part["posting_date"].max())
print("validation:", validation_part.shape, validation_part["posting_date"].min(), validation_part["posting_date"].max())


Какое свойство будущего test воспроизводит это разбиение, а какое найденное различие пока не воспроизводит?


## Проверка 4. Первый честный ориентир

Соберем линейную модель на нескольких числовых и категориальных признаках. Числовые признаки стандартизируем, а категории представим с помощью one-hot кодирования.

Все преобразования обучаются только на train. Validation получает уже найденные средние, масштабы и категории.


In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def fit_tabular_encoder(table, numeric_features, categorical_features):
    scaler = StandardScaler()
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)

    numeric = scaler.fit_transform(table[numeric_features])
    categorical = encoder.fit_transform(table[categorical_features].astype(str))
    matrix = hstack([numeric, categorical], format="csr")

    state = {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "scaler": scaler,
        "encoder": encoder,
    }
    return matrix, state


In [ ]:
def transform_tabular(table, state):
    numeric = state["scaler"].transform(table[state["numeric_features"]])
    categorical = state["encoder"].transform(
        table[state["categorical_features"]].astype(str)
    )
    return hstack([numeric, categorical], format="csr")


def evaluate_tabular(train_table, validation_table, numeric_features, categorical_features, alpha=10):
    X_train, state = fit_tabular_encoder(
        train_table, numeric_features, categorical_features
    )
    X_validation = transform_tabular(validation_table, state)

    model = Ridge(alpha=alpha, solver="lsqr")
    model.fit(X_train, train_table[TARGET])

    return {
        "train_rmse": rmse(train_table[TARGET], model.predict(X_train)),
        "validation_rmse": rmse(
            validation_table[TARGET], model.predict(X_validation)
        ),
        "model": model,
        "state": state,
        "X_train": X_train,
        "X_validation": X_validation,
    }


Для первого ориентира возьмем небольшой общий набор признаков. `job_id`, дату и описание пока не используем. Модель стандартизирует числа и кодирует категории с помощью one-hot.


In [ ]:
# Посчитайте train и validation RMSE для заданного набора признаков.
# Сохраните результат evaluate_tabular как baseline.
BASE_NUMERIC = ["rating"]
BASE_CATEGORICAL = [
    "job_title",
    "industry",
    "sector",
    "company_size",
    "ownership_type",
    "location",
]


Сравните train и validation RMSE. Для чего понадобится эта пара чисел в следующих экспериментах?


## Идея 1. Как представить дату

Средняя зарплата по месяцам показывает форму временной зависимости до обучения модели. Ячейка ниже считает среднее только на `train_part`.

До запуска предположите, какую форму зависимости от времени вообще может увидеть линейная модель.

<details>
<summary>Подсказка</summary>

Из даты можно получить месяц вызовом `table["posting_date"].dt.to_period("M")`. Затем пригодятся `groupby`, `mean` и `plot`.

</details>


In [ ]:
monthly_salary = (
    train_part
    .assign(month=train_part["posting_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month", as_index=False)[TARGET]
    .mean()
)

plt.plot(monthly_salary["month"], monthly_salary[TARGET], marker="o")
plt.axvline(pd.Timestamp("2025-02-01"), color="tab:red", linestyle="--", label="validation")
plt.ylabel("Средняя зарплата, тыс. долларов")
plt.xlabel("Месяц публикации")
plt.legend()
plt.show()


Посмотрите на график и назовите хотя бы две разные закономерности, которые могут быть связаны с датой. Затем решите, достаточно ли для них одного возрастающего числа.

Запишем дату одним числом: количеством дней с 1 января 2023 года. До запуска предположите результат, затем сравните его с базовой моделью.

<details>
<summary>Подсказка</summary>

Из столбца с датой можно вычесть `pd.Timestamp("2023-01-01")`, а затем взять число дней через `.dt.days`.

</details>


In [ ]:
train_with_date = train_part.copy()
validation_with_date = validation_part.copy()

start_date = pd.Timestamp("2023-01-01")
train_with_date["date_number"] = (
    train_with_date["posting_date"] - start_date
).dt.days
validation_with_date["date_number"] = (
    validation_with_date["posting_date"] - start_date
).dt.days

raw_date_result = evaluate_tabular(
    train_with_date,
    validation_with_date,
    BASE_NUMERIC + ["date_number"],
    BASE_CATEGORICAL,
)

print("Без даты:", round(baseline["validation_rmse"], 3))
print("Дата одним числом:", round(raw_date_result["validation_rmse"], 3))


Какую зависимость способна выразить линейная модель с одним числовым признаком даты? Согласуется ли с ней форма графика?


### Циклическая часть даты

Какую идею из лекции можно применить, если часть зависимости повторяется по календарю? Вспомните, почему обычный номер месяца не передает соседство декабря и января.

Для проверки используем пару циклических признаков:

Разместим месяцы на окружности:

$$
\text{month\_sin}=\sin\left(2\pi\frac{m-1}{12}\right),\qquad
\text{month\_cos}=\cos\left(2\pi\frac{m-1}{12}\right).
$$

Одного синуса недостаточно: одинаковому значению соответствуют две разные точки окружности. Пара синуса и косинуса задает положение месяца однозначно.


In [ ]:
# Создайте month_sin и month_cos для train и validation.
# Не добавляйте date_number.
# Сравните validation RMSE с двумя предыдущими моделями.


Сравните три строки таблицы. Какая гипотеза о дате получила поддержку, а какая пока нет? Не делайте общий вывод «дата полезна» или «дата бесполезна»: разные представления проверяют разные идеи.


Сформулируйте вывод по трем строкам таблицы.


### Два наклона вместо одного

Вернитесь к графику. Какие новые числа можно получить из даты, чтобы линейная модель могла вести себя по-разному до и после заметного изменения?

<details>
<summary>Открыть один вариант конструкции</summary>

На графике изменение начинается примерно в ноябре 2024 года. Введем два признака:

$$
\text{month\_index}=0,1,2,\ldots
$$

$$
\text{months\_after\_turn}=\max(\text{month\_index}-22,0).
$$

До разворота второй признак равен нулю. После разворота он растет, поэтому модель может выучить новый наклон. Это нелинейное преобразование даты, но сама модель остается линейной по созданным признакам.

</details>

<details>
<summary>Подсказка</summary>

Непрерывный номер месяца можно получить так: `(year - 2023) * 12 + month - 1`. Положительную часть разности удобно взять методом `.clip(lower=0)`.

</details>


In [ ]:
# Добавьте month_index и months_after_turn к таблицам с sin/cos.
# Сравните тренд отдельно и тренд вместе с sin/cos.
# Сохраните выбранные признаки даты в DATE_FEATURES.


Сравните результат с циклическими признаками. Что именно добавила новая конструкция? Небольшое изменение одной validation еще не доказывает, что прием всегда полезен. На других данных точку изменения придется заново обосновать и проверить.


Сформулируйте, какой новый вид зависимости получила модель.


## Идея 2. Признаки из заголовка и описания

Посмотрите на несколько заголовков и описаний. До кода придумайте не меньше четырех гипотез о том, какие свойства вакансии могут быть связаны с зарплатой. Постарайтесь взять идеи из разных источников: заголовка, текста, географии или сведений о компании.

Для каждой идеи сначала сформулируйте признак обычными словами, а затем решите, как превратить его в число. Напишите функцию `add_manual_features`, которая вернет копию таблицы с выбранными новыми столбцами.

<details>
<summary>Подсказка</summary>

Для работы со строками пригодятся раздел [Text data](https://pandas.pydata.org/docs/user_guide/text.html) и методы `.str`. Если числовой признак имеет очень длинный хвост, вспомните о преобразованиях масштаба из лекции.

</details>


In [ ]:
def add_manual_features(table):
    result = table.copy()

    # Добавьте признаки.

    return result


In [ ]:
# Сохраните таблицы как train_manual и validation_manual.
# Сохраните названия новых столбцов в MANUAL_FEATURES.
# Сохраните результат обучения как manual_result.
# Проверьте вклад каждого нового признака удалением из общей модели.

MANUAL_FEATURES = [
    # ...
]


Новый признак считается полезным не потому, что звучит разумно, а потому, что улучшает одну и ту же честную validation. Несколько признаков добавлены одновременно, поэтому метрика оценивает всю группу. Вклад отдельной идеи проверяется отдельным экспериментом.


Сформулируйте вывод по группе ручных признаков.


## Идея 3. TF-IDF и регуляризация

Ручные признаки проверяют только заранее придуманные слова. TF-IDF строит признак для каждого слова или сочетания слов.

Пусть слово встретилось в документе несколько раз. Его частота внутри документа называется TF. Но частое слово не всегда полезно: слово `data` может встречаться почти в каждой вакансии. Поэтому частоту умножают на IDF, которая уменьшается для слов, встречающихся во многих документах:

$$
\operatorname{IDF}(t)=\log\frac{N+1}{\operatorname{df}(t)+1}+1,
\qquad
\operatorname{TFIDF}(t,d)=\operatorname{TF}(t,d)\cdot\operatorname{IDF}(t).
$$

Здесь `N` - число документов, а `df(t)` - число документов со словом `t`. После этого каждая строка обычно нормируется по L2-норме. Матрица получается разреженной: почти все ее элементы равны нулю, поэтому превращать ее в обычный массив через `toarray()` не нужно.

Параметры, которые особенно полезно понимать:

- `min_df` удаляет слишком редкие слова;
- `max_df` удаляет слова почти из всех документов;
- `max_features` ограничивает размер словаря;
- `ngram_range=(1, 2)` добавляет пары соседних слов;
- `analyzer="char"` строит символьные сочетания и может быть устойчивее к опечаткам.


Склейте заголовок и описание. Обучите `TfidfVectorizer` только на текстах train, затем примените его к validation.

Подберите разумные `min_df`, `ngram_range` и `max_features`. Для первого запуска хватит слов и пар слов. Сохраните параметры в словаре `TFIDF_PARAMS`: он понадобится при финальном обучении.

<details>
<summary>Подсказка</summary>

Документация: [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) и [пример классификации текста](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html).

Нужная последовательность: создать векторизатор, вызвать `fit_transform` на train и `transform` на validation. Не вызывайте `fit` на validation.

</details>


In [ ]:
train_text = (
    train_manual["job_title"].fillna("")
    + " "
    + train_manual["job_description"].fillna("")
)
validation_text = (
    validation_manual["job_title"].fillna("")
    + " "
    + validation_manual["job_description"].fillna("")
)

# Сохраните выбранные параметры в TFIDF_PARAMS.
# Создайте vectorizer через TfidfVectorizer(**TFIDF_PARAMS).
# Получите X_text_train и X_text_validation.


In [ ]:
assert X_text_train.shape[0] == len(train_manual)
assert X_text_validation.shape[0] == len(validation_manual)
assert X_text_train.shape[1] == X_text_validation.shape[1]


Соединим табличные и текстовые признаки. У текста тысячи столбцов, многие из них похожи друг на друга. L2-регуляризация может сделать веса устойчивее, но не обязана улучшать каждую validation.

Заголовок вакансии теперь представлен дважды: one-hot категорией и набором слов. Это осознанный эксперимент, а не обязательное правило.

В сравнение обязательно включим `alpha = 0`: это Ridge без L2-штрафа. До запуска предскажите, как будут меняться train и validation RMSE. Затем выберите `alpha` по validation, даже если лучшим окажется нуль.


In [ ]:
X_train_all = hstack(
    [manual_result["X_train"], X_text_train], format="csr"
)
X_validation_all = hstack(
    [manual_result["X_validation"], X_text_validation], format="csr"
)

text_results = []
for alpha in [0, 0.01, 0.1, 0.3, 1, 3, 10]:
    model = Ridge(alpha=alpha, solver="lsqr")
    model.fit(X_train_all, train_manual[TARGET])

    text_results.append(
        {
            "alpha": alpha,
            "train_rmse": rmse(train_manual[TARGET], model.predict(X_train_all)),
            "validation_rmse": rmse(
                validation_manual[TARGET], model.predict(X_validation_all)
            ),
        }
    )

text_results = pd.DataFrame(text_results)
display(text_results)


In [ ]:
# Сохраните лучшее alpha по validation.
BEST_ALPHA = ...


Объясните выбор `alpha` через обе колонки с ошибками. Если выиграло `alpha = 0`, так и запишите: регуляризация не обязана помогать на каждой validation.


## Возвращаемся к сдвигу train и test

Классификатор источника особенно выделил `company_size` и `rating`. Это не означает, что оба признака нужно удалить. Проверим три версии одной и той же модели:

1. все признаки;
2. без `company_size`;
3. без `rating`.

Сопоставьте изменение validation RMSE с силой сдвига. Иногда ради устойчивости разумно принять небольшое ухудшение, но большую потерю нельзя игнорировать только из-за опасения сдвига.


In [ ]:
# Обучите три текстовые модели с BEST_ALPHA.
# Меняйте только набор числовых или категориальных признаков.
# Соберите validation RMSE в shift_ablation.
# После сравнения сохраните финальные списки именно в
# FINAL_NUMERIC и FINAL_CATEGORICAL: их используют следующие ячейки.

FINAL_NUMERIC = ...
FINAL_CATEGORICAL = ...


Какой вывод из анализа сдвига действительно подтверждается задачей прогноза зарплаты?


## Финальная проверка

Решение зафиксировано: четыре признака даты, четыре ручных признака, TF-IDF, `rating`, категориальные признаки без `company_size` и Ridge с выбранным `alpha`.

Все обучаемые преобразования получают `fit` на полном train. К test применяется только `transform`, после чего модель строит итоговый прогноз.

Нельзя менять признаки и параметры после просмотра настоящих ответов test.


In [ ]:
def add_date_features(table):
    result = table.copy()
    month = result["posting_date"].dt.month
    angle = 2 * np.pi * (month - 1) / 12
    result["month_sin"] = np.sin(angle)
    result["month_cos"] = np.cos(angle)
    result["month_index"] = (
        (result["posting_date"].dt.year - 2023) * 12
        + result["posting_date"].dt.month
        - 1
    )
    result["months_after_turn"] = (result["month_index"] - 22).clip(lower=0)
    return result


final_train = add_manual_features(add_date_features(train_clean))
final_test = add_manual_features(add_date_features(test_clean))


In [ ]:
X_final_tabular, final_state = fit_tabular_encoder(
    final_train,
    FINAL_NUMERIC,
    FINAL_CATEGORICAL,
)
X_test_tabular = transform_tabular(final_test, final_state)


In [ ]:
final_train_text = (
    final_train["job_title"].fillna("")
    + " "
    + final_train["job_description"].fillna("")
)
final_test_text = (
    final_test["job_title"].fillna("")
    + " "
    + final_test["job_description"].fillna("")
)


In [ ]:
final_vectorizer = TfidfVectorizer(**TFIDF_PARAMS)
X_final_text = final_vectorizer.fit_transform(final_train_text)
X_test_text = final_vectorizer.transform(final_test_text)


In [ ]:
X_final = hstack([X_final_tabular, X_final_text], format="csr")
X_test = hstack([X_test_tabular, X_test_text], format="csr")

final_model = Ridge(alpha=BEST_ALPHA, solver="lsqr")
final_model.fit(X_final, final_train[TARGET])
test_predictions = final_model.predict(X_test)


In [ ]:
assert len(test_predictions) == len(test_clean)
assert np.isfinite(test_predictions).all()

predictions = pd.DataFrame(
    {
        "job_id": test_clean["job_id"],
        "salary_k_usd": test_predictions,
    }
)
display(predictions.head())


<div class="warning">
<strong>Открывайте следующий файл только после фиксации решения.</strong><br>
Мы симулируем олимпиаду. После проверки test нельзя возвращаться назад, менять признаки, параметры или выбирать другую модель по этой оценке.
</div>


### Однократная проверка на будущем test

Только сейчас можно получить отдельную ссылку на `test_with_labels.csv` с настоящими ответами. Вставьте присланную ссылку в следующую ячейку.


In [ ]:
LABELS_URL = ""  # Вставьте ссылку, которую преподаватель пришлет в конце.

if not LABELS_URL:
    raise ValueError("Сначала вставьте ссылку на test_with_labels.csv")

test_with_labels = pd.read_csv(LABELS_URL)
checked = predictions.merge(
    test_with_labels[["job_id", TARGET]],
    on="job_id",
    how="inner",
    suffixes=("_prediction", "_true"),
    validate="one_to_one",
)
test_rmse = rmse(
    checked[f"{TARGET}_true"],
    checked[f"{TARGET}_prediction"],
)
assert len(checked) == len(test_clean)
print("test RMSE:", round(test_rmse, 3))


Зафиксируйте, насколько test согласуется с validation. Не меняйте решение по этой оценке.


## Задания со звездочкой

Эти задания продолжают основную практику. Для каждого сначала сформулируйте гипотезу и выберите validation, на которой ее можно честно проверить.


In [ ]:
from scipy.sparse import vstack

from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)


### 1. Символьный TF-IDF

Постройте TF-IDF по сочетаниям символов и сравните три варианта на той же временной validation: только слова, только символы, слова и символы вместе. Подумайте, какие опечатки, окончания и короткие обозначения символьное представление заметит лучше.

<details>
<summary>Подсказка</summary>

Документация: [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).

Обратите внимание на `analyzer="char_wb"` и `ngram_range`. После `fit_transform` и `transform` соедините текст с `manual_result["X_train"]` и `manual_result["X_validation"]` через `hstack`.

</details>


In [ ]:
# Создайте char_vectorizer.
# Получите X_char_train и X_char_validation.
# Обучите Ridge для char и для word + char.
# Сравните обе модели со словесным TF-IDF.


### 2. Новые компании в validation

Разделите строки так, чтобы одна компания целиком попадала либо в train, либо в validation. Сравните такой результат с временной validation и объясните, какие два разных вида переноса проверяют эти разбиения.

<details>
<summary>Подсказка</summary>

Документация: [`GroupShuffleSplit`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html).

В `groups` передайте `company_name`. После разбиения проверьте пересечение множеств компаний через `set`.

</details>


In [ ]:
# Получите group_train и group_validation.
# Проверьте, что компании не пересекаются.
# Повторите обучение основной модели на новом split.


### 3. Повторяющиеся описания

Сделайте случайное разбиение того же размера, что и временная validation. Посчитайте, сколько точных копий `job_description` оказалось с обеих сторон. Затем отдельно измерьте RMSE на знакомых и новых описаниях.

<details>
<summary>Подсказка</summary>

Документация: [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) и [`Series.isin`](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html).

Для числа общих описаний можно пересечь два `set`. Булева маска `isin` позволит разделить validation на знакомые и новые тексты.

</details>


In [ ]:
# Создайте random_train и random_validation.
# Найдите точные копии описаний между частями.
# Посчитайте общий RMSE и два RMSE по маске знакомых описаний.


### 4. Ridge и Lasso

Обучите Lasso с несколькими значениями `alpha`. Для каждого значения сохраните train RMSE, validation RMSE и число ненулевых коэффициентов. Сравните результат с Ridge.

<details>
<summary>Подсказка</summary>

Документация: [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) и [`numpy.count_nonzero`](https://numpy.org/doc/stable/reference/generated/numpy.count_nonzero.html).

Матрицу можно передать в формате CSC через `.tocsc()`. Число ненулевых весов находится по `model.coef_`.

</details>


In [ ]:
# Проверьте несколько alpha для Lasso.
# Соберите таблицу train_rmse, validation_rmse и nonzero_weights.


### 5. Среднее по отрасли без утечки

Постройте для каждой обучающей строки среднюю зарплату в ее отрасли только по более ранним датам. Строки одной даты не должны видеть ответы друг друга. Для validation используйте статистику, рассчитанную только по train. Проверьте признак и в табличной модели, и вместе с TF-IDF.

<details>
<summary>Подсказка</summary>

Документация: [`groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html), [`cumsum`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.cumsum.html) и [`merge`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html).

Сначала агрегируйте сумму и число вакансий по `industry` и `posting_date`. Кумулятивная сумма до текущей даты равна общей кумулятивной сумме минус значение текущей даты.

</details>


In [ ]:
def add_past_industry_mean(train_table, validation_table):
    # Верните две копии с признаком industry_salary_past.
    # Не используйте текущие и будущие ответы.
    pass


### 6. Логарифм зарплаты

Обучите Ridge предсказывать `log(1 + target)`. Верните прогноз в исходную шкалу и посчитайте RMSE в тысячах долларов. Сравните его с моделью без преобразования target.

<details>
<summary>Подсказка</summary>

Документация: [`numpy.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html) и [`numpy.expm1`](https://numpy.org/doc/stable/reference/generated/numpy.expm1.html).

Модель обучается на `np.log1p(y_train)`, а перед расчетом RMSE прогноз преобразуется через `np.expm1`.

</details>


In [ ]:
# Обучите log_model на логарифме зарплаты.
# Получите log_predictions в исходной шкале и log_target_rmse.


### 7. MAE и RMSE

Посчитайте MAE и RMSE для одного и того же прогноза. Найдите вакансии с крупнейшими абсолютными ошибками и объясните, почему они сильнее влияют на одну из метрик.

<details>
<summary>Подсказка</summary>

Документация: [`mean_absolute_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html) и [`sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html).

Добавьте к таблице validation прогноз, абсолютную ошибку и квадрат ошибки.

</details>


In [ ]:
# Посчитайте validation_mae и validation_rmse.
# Выведите вакансии с самыми большими absolute_error.


### 8. Коэффициенты модели

Соберите названия числовых, категориальных и текстовых признаков. Найдите самые большие положительные и отрицательные коэффициенты Ridge. Отделите наблюдаемую связь от причинного вывода.

<details>
<summary>Подсказка</summary>

Документация: [`OneHotEncoder.get_feature_names_out`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html#sklearn.preprocessing.OneHotEncoder.get_feature_names_out) и [`TfidfVectorizer.get_feature_names_out`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html#sklearn.feature_extraction.text.TfidfVectorizer.get_feature_names_out).

Порядок названий должен совпадать с порядком матриц внутри `hstack`. Проверьте длину списка через `assert`.

</details>


In [ ]:
# Соберите feature_names в правильном порядке.
# Создайте таблицу feature, coefficient.
# Выведите оба края таблицы коэффициентов.


### 9. Adversarial validation

Обучите классификатор отличать ранние вакансии от поздних. Прямые признаки даты исключите, иначе модель просто прочитает ответ из календаря. Оцените ROC AUC, затем изучите коэффициенты и назовите конкретные признаки сдвига.

<details>
<summary>Подсказка</summary>

Документация: [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) и [`roc_auc`](https://scikit-learn.org/stable/modules/model_evaluation.html#roc-metrics).

Соедините раннюю и позднюю матрицы через `vstack`. Метка периода равна `0` для train и `1` для validation. Используйте `StratifiedKFold`.

</details>


In [ ]:
# Соберите признаки без DATE_FEATURES.
# Создайте y_adversarial из нулей и единиц.
# Посчитайте ROC AUC по фолдам.
# Обучите модель на всех строках и выведите крайние коэффициенты.
